#### Task 2: Data Quality Audit on Top of an ETL Pipeline

This notebook fetches 100 posts from JSONPlaceholder, audits the raw data before any cleaning, applies the required transformations, generates a structured audit report, and loads the clean dataset into MySQL.

In [110]:
from pathlib import Path

import mysql.connector
import pandas as pd
import requests
from IPython.display import display

API_URL = "https://jsonplaceholder.typicode.com/posts"
RAW_LIMIT = 100
AUDIT_REPORT_PATH = Path.cwd() / "task_2_audit_report.csv"
EXPECTED_SCHEMA = {"userId": "int64", "id": "int64", "title": "string", "body": "string"}


def fetch_posts(limit=RAW_LIMIT):
    response = requests.get(
        API_URL,
        params={"_limit": limit},
        timeout=20
    )
    response.raise_for_status()
    data = response.json()
    df = pd.DataFrame(data)
    df = df[['userId', 'id', 'title', 'body']]
    return df

### Audit

In [111]:
def build_pre_clean_audit(df):

    # 1. NULL VALUE CHECK

    # Count null values in each column
    null_counts = df.isnull().sum()
    print("\nNull Values:")
    print(null_counts)


    # 2. DUPLICATE ROW CHECK

    duplicate_row_count = df.duplicated().sum()
    print("\nDuplicate Rows:")
    print(duplicate_row_count)


    # 3. DATA TYPE CHECK

    print("\nData Type Check:")

    type_issues = []

    # Check userId column
    if not pd.api.types.is_integer_dtype(df['userId']):
        type_issues.append("userId column is not integer")


    # Check id column
    if not pd.api.types.is_integer_dtype(df['id']):
        type_issues.append("id column is not integer")


    # Check title column
    if not pd.api.types.is_string_dtype(df['title']):
        type_issues.append("title column is not string")


    # Check body column
    if not pd.api.types.is_string_dtype(df['body']):
        type_issues.append("body column is not string")


    # Print type issues
    if len(type_issues) == 0:
        print("No datatype issues found")
    else:
        for issue in type_issues:
            print(issue)


    # 4. OUTLIER / RANGE CHECK


    print("\nRange Check:")
    # userId should be between 1 and 10
    invalid_user_ids = (~df['userId'].between(1, 10)).sum()
    print("Invalid userId count:", invalid_user_ids)


    # id should be between 1 and 100
    invalid_ids = (~df['id'].between(1, 100)).sum()
    print("Invalid id count:", invalid_ids)


    # 5. STRING FORMAT CHECK

    print("\nString Format Check:")

    # Remove spaces
    cleaned_titles = df['title'].fillna("").astype(str).str.strip()

    # Find inconsistent title formatting
    wrong_format_mask = cleaned_titles != cleaned_titles.str.title()

    inconsistent_titles = df.loc[
        wrong_format_mask,
        ['id', 'title']
    ]

    print(inconsistent_titles.head(10))


    # 6. FINAL ISSUE SUMMARY

    total_nulls = df.isnull().sum().sum()
    total_type_issues = len(type_issues)
    total_range_issues = invalid_user_ids + invalid_ids
    total_format_issues = wrong_format_mask.sum()

    total_issues = (
        total_nulls
        + duplicate_row_count
        + total_type_issues
        + total_range_issues
        + total_format_issues
    )

    print("\n------------ ISSUE SUMMARY ------------")

    print("Total Null Values:", total_nulls)
    print("Duplicate Rows:", duplicate_row_count)
    print("Datatype Issues:", total_type_issues)
    print("Range Issues:", total_range_issues)
    print("Formatting Issues:", total_format_issues)
    print("Total Issues Found:", total_issues)


    # RETURN RESULT
    issue_summary = pd.DataFrame([
        {"issue_type": "Null values", "detected_count": total_nulls},
        {"issue_type": "Duplicate rows", "detected_count": duplicate_row_count},
        {"issue_type": "Type mismatches", "detected_count": total_type_issues},
        {"issue_type": "Out-of-range values", "detected_count": total_range_issues},
        {"issue_type": "Inconsistent string formats", "detected_count": total_format_issues},
    ])

    return {
        "null_counts": null_counts,
        "type_checks": pd.DataFrame([{"column": "userId", "expected_type": "int64", "actual_type": str(df['userId'].dtype), "mismatch": not pd.api.types.is_integer_dtype(df['userId'])}, {"column": "id", "expected_type": "int64", "actual_type": str(df['id'].dtype), "mismatch": not pd.api.types.is_integer_dtype(df['id'])}, {"column": "title", "expected_type": "string", "actual_type": str(df['title'].dtype), "mismatch": not pd.api.types.is_string_dtype(df['title'])}, {"column": "body", "expected_type": "string", "actual_type": str(df['body'].dtype), "mismatch": not pd.api.types.is_string_dtype(df['body'])}]),
        "out_of_range_checks": pd.DataFrame([{"column": "userId", "rule": "between 1 and 10", "issue_count": invalid_user_ids}, {"column": "id", "rule": "between 1 and 100", "issue_count": invalid_ids}]),
        "inconsistent_string_examples": inconsistent_titles.head(10),
        "issue_summary": issue_summary,
        "total_issues": total_issues
    }

### Clean

In [112]:
def clean_posts(df):

    # Create copy so original dataframe remains unchanged
    cleaned = df.copy()

    # Clean title column
    cleaned['title'] = (
        cleaned['title']
        .fillna("")
        .astype(str)
        .str.strip()
        .str.title()
    )

    # Clean body column
    cleaned['body'] = (
        cleaned['body']
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Store original cleaned titles for comparison
    original_titles = (
        df['title']
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Store updated titles
    fixed_titles = cleaned['title']

    # Count how many titles were fixed
    title_fixed_count = (
        original_titles != fixed_titles
    ).sum()

    # Remove rows with empty title or body
    cleaned = cleaned[
        (cleaned['title'] != "") &
        (cleaned['body'] != "")
    ]

    # Count rows before removing duplicates
    before_duplicates = len(cleaned)

    # Remove duplicate rows based on id
    cleaned = cleaned.drop_duplicates(
        subset=['id']
    )

    # Count rows after removing duplicates
    after_duplicates = len(cleaned)

    # Calculate number of duplicates removed
    duplicates_removed = (
        before_duplicates - after_duplicates
    )

    # Count rows before range filtering
    before_range = len(cleaned)

    # Keep only valid userId and id ranges
    cleaned = cleaned[
        cleaned['userId'].between(1, 10) &
        cleaned['id'].between(1, 100)
    ]

    # Count rows after filtering
    after_range = len(cleaned)

    # Calculate removed out-of-range rows
    out_of_range_removed = (
        before_range - after_range
    )

    # Count words in title
    title_words = (
        cleaned['title'].str.split().str.len()
    )

    # Count words in body
    body_words = (
        cleaned['body'].str.split().str.len()
    )

    # Create total word count column
    cleaned['word_count'] = (
        title_words + body_words
    )

    # Create ranking based on word count
    cleaned['title_rank'] = (
        cleaned['word_count']
        .rank(
            method='dense',
            ascending=False
        )
        .astype(int)
    )

    # Sort dataframe
    cleaned = cleaned.sort_values(
        ['title_rank', 'id']
    )

    # Reset row index
    cleaned = cleaned.reset_index(drop=True)

    # Store cleaning summary
    fix_summary = {

        "title_fixed_count": title_fixed_count,
        "duplicates_removed": duplicates_removed,
        "out_of_range_removed": out_of_range_removed,
        "type_coercions": 0
    }

    # Return cleaned dataframe and summary
    return cleaned, fix_summary


def build_audit_report(
    pre_clean_audit,
    fix_summary,
    before_rows,
    after_rows
):

    # Copy issue summary dataframe
    report = pre_clean_audit['issue_summary'].copy()

    # Add fixed issue counts
    report['fixed_count'] = [
        0,

        fix_summary['duplicates_removed'],
        fix_summary['type_coercions'],
        fix_summary['out_of_range_removed'],
        fix_summary['title_fixed_count']
    ]

    # Add total rows before cleaning
    report['before_row_count'] = before_rows

    # Add total rows after cleaning
    report['after_row_count'] = after_rows

    # Return final audit report
    return report

### Load

In [113]:
def load_to_mysql(df):

    # Store MySQL connection settings
    mysql_config = {

        "host": os.getenv("MYSQL_HOST", "localhost"),
        "port": int(os.getenv("MYSQL_PORT", "3306")),
        "user": os.getenv("MYSQL_USER", "root"),
        "password": os.getenv("MYSQL_PASSWORD", "root"),
        "database": os.getenv("MYSQL_DATABASE", "week5_audit"),
    }

    # Connect to MySQL server
    connection = mysql.connector.connect(
        host=mysql_config["host"],
        port=mysql_config["port"],
        user=mysql_config["user"],
        password=mysql_config["password"],
    )

    # Create cursor object
    cursor = connection.cursor()

    # Create database if it does not exist
    cursor.execute(
        f"CREATE DATABASE IF NOT EXISTS `{mysql_config['database']}`"
    )

    # Select database
    cursor.execute(
        f"USE `{mysql_config['database']}`"
    )

    # Create table if it does not exist
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS clean_posts (
            id INT PRIMARY KEY,
            userId INT NOT NULL,
            title VARCHAR(255) NOT NULL,
            body TEXT NOT NULL,
            word_count INT NOT NULL,
            title_rank INT NOT NULL
        )
        """
    )

    # SQL query for insert/update
    insert_sql = """

        INSERT INTO clean_posts (
            id,
            userId,
            title,
            body,
            word_count,
            title_rank

        )

        VALUES (%s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            userId = VALUES(userId),
            title = VALUES(title),
            body = VALUES(body),
            word_count = VALUES(word_count),
            title_rank = VALUES(title_rank)
    """

    # Convert dataframe rows into tuple format
    rows = list(
        df[
            [
                "id",
                "userId",
                "title",
                "body",
                "word_count",
                "title_rank"
            ]
        ].itertuples(
            index=False,
            name=None
        )
    )

    # Insert multiple rows into database
    cursor.executemany(insert_sql, rows)

    connection.commit()
    cursor.close()
    connection.close()

    # Return number of inserted rows
    return len(rows)

In [114]:
raw_posts = fetch_posts()
raw_posts.head()
print(f"Fetched {len(raw_posts)} rows and {len(raw_posts.columns)} columns from JSONPlaceholder.")
print(raw_posts.dtypes)

Fetched 100 rows and 4 columns from JSONPlaceholder.
userId    int64
id        int64
title       str
body        str
dtype: object


In [115]:
pre_clean_audit = build_pre_clean_audit(raw_posts)

print("Null counts by column:")
display(pre_clean_audit["null_counts"])

print("Type checks:")
display(pre_clean_audit["type_checks"])

print("Out-of-range checks:")
display(pre_clean_audit["out_of_range_checks"])

print("Examples of inconsistent string formats:")
display(pre_clean_audit["inconsistent_string_examples"])

print("Issue summary before cleaning:")
display(pre_clean_audit["issue_summary"])
print(f"Total issues detected before cleaning: {pre_clean_audit['total_issues']}")


Null Values:
userId    0
id        0
title     0
body      0
dtype: int64

Duplicate Rows:
0

Data Type Check:
No datatype issues found

Range Check:
Invalid userId count: 0
Invalid id count: 0

String Format Check:
   id                                                                       title
0   1  sunt aut facere repellat provident occaecati excepturi optio reprehenderit
1   2                                                                qui est esse
2   3                 ea molestias quasi exercitationem repellat qui ipsa sit aut
3   4                                                        eum et est occaecati
4   5                                                          nesciunt quas odio
5   6                                          dolorem eum magni eos aperiam quia
6   7                                                        magnam facilis autem
7   8                                                    dolorem dolore est ipsam
8   9                          nesciunt iure 

userId    0
id        0
title     0
body      0
dtype: int64

Type checks:


,column,expected_type,actual_type,mismatch
0,userId,int64,int64,False
1,id,int64,int64,False
2,title,string,str,False
3,body,string,str,False


Out-of-range checks:


...

Examples of inconsistent string formats:


,id,title
0,1,sunt aut facere repellat provident occaecati excepturi optio reprehenderit
1,2,qui est esse
2,3,ea molestias quasi exercitationem repellat qui ipsa sit aut
3,4,eum et est occaecati
4,5,nesciunt quas odio
5,6,dolorem eum magni eos aperiam quia
6,7,magnam facilis autem
7,8,dolorem dolore est ipsam
8,9,nesciunt iure omnis dolorem tempora et accusantium
9,10,optio molestias id quia eum


Issue summary before cleaning:


,issue_type,detected_count
0,Null values,0
1,Duplicate rows,0
2,Type mismatches,0
3,Out-of-range values,0
4,Inconsistent string formats,100


Total issues detected before cleaning: 100


In [116]:
clean_posts_df, fix_summary = clean_posts(raw_posts)
clean_posts_df.head()
print(f"Rows before cleaning: {len(raw_posts)}")
print(f"Rows after cleaning: {len(clean_posts_df)}")
print(fix_summary)

Rows before cleaning: 100
Rows after cleaning: 100
{'title_fixed_count': np.int64(100), 'duplicates_removed': 0, 'out_of_range_removed': 0, 'type_coercions': 0}


In [117]:
audit_report = build_audit_report(pre_clean_audit, fix_summary, len(raw_posts), len(clean_posts_df))

audit_report.to_csv(AUDIT_REPORT_PATH, index=False)
print(f"Audit report saved to: {AUDIT_REPORT_PATH}")
print(f"Total issues detected: {pre_clean_audit['total_issues']}")
print(f"Total issues fixed: {int(audit_report['fixed_count'].sum())}")
print(f"Before row count: {len(raw_posts)}")
print(f"After row count: {len(clean_posts_df)}")
display(audit_report)

Audit report saved to: c:\Users\PM\Desktop\Internship\AI-Data-Engineering-Internship\Week-5 May_12\task_2_audit_report.csv
Total issues detected: 100
Total issues fixed: 100
Before row count: 100
After row count: 100


,issue_type,detected_count,fixed_count,before_row_count,after_row_count
0,Null values,0,0,100,100
1,Duplicate rows,0,0,100,100
2,Type mismatches,0,0,100,100
3,Out-of-range values,0,0,100,100
4,Inconsistent string formats,100,100,100,100


## MySQL Load

In [118]:
try:
    rows_loaded = load_to_mysql(clean_posts_df)
    print(f"Rows written to MySQL: {rows_loaded}")
except mysql.connector.Error as exc:
    print("MySQL load could not be completed in this environment.")
    print("Set the MySQL environment variables and make sure the server is running, then rerun this cell.")
    print(exc)

Rows written to MySQL: 100
